# An Implementation of Modification of safPAKE  #

<h2> This Notebook gives an actual benchmark of our BIO-PAKE </h2>

In [1]:
import face_recognition
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import pandas as pd
from bsp import CosineLSH
import json
import python_bulletproofs

In [2]:
import time
import csv
import statistics
import random

# 1 Lib Sodium Initialization and some EC-Operation Functions #

In [ ]:
import ctypes
import ctypes.util
import os
import hashlib
import secrets
from typing import List, Tuple

# --- 1. Libsodium Loading & Bindings ---

SCALAR_LEN = 32
POINT_LEN = 32

def load_sodium():
    for name in ("sodium", "libsodium"):
        path = ctypes.util.find_library(name)
        if path:
            try: return ctypes.CDLL(path)
            except: pass
    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates = [
            os.path.join(conda_prefix, "Library", "bin", "libsodium.dll"),
            os.path.join(conda_prefix, "lib", "libsodium.so"),
        ]
        for c in candidates:
            if os.path.exists(c): return ctypes.CDLL(c)
    raise OSError("libsodium not found")

sodium = load_sodium()
if hasattr(sodium, "sodium_init"):
    sodium.sodium_init()

# --- Bindings ---

# Scalar Math
sodium.crypto_core_ristretto255_scalar_random.argtypes = [ctypes.c_void_p]
try:
    sodium.crypto_core_ristretto255_scalar_mul.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_scalar_mul.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing scalar_mul).")
sodium.crypto_core_ristretto255_scalar_invert.argtypes = [ctypes.c_void_p, ctypes.c_void_p]

# Point Math
sodium.crypto_scalarmult_ristretto255_base.argtypes = [ctypes.c_void_p, ctypes.c_void_p] # P = n * G
sodium.crypto_scalarmult_ristretto255.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p] # P = n * Q

# NEW: Point Addition and Subtraction
try:
    # R = P + Q
    sodium.crypto_core_ristretto255_add.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_add.restype = ctypes.c_int
    
    # R = P - Q
    sodium.crypto_core_ristretto255_sub.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_sub.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing point add/sub).")


# --- Wrappers ---

def random_scalar() -> bytes:
    buf = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_random(ctypes.byref(buf))
    return bytes(buf)

def random_point() -> bytes:
    # To get a random valid point, we generate a random scalar and multiply by base
    return scalar_to_point(random_scalar())

def scalar_mul(x: bytes, y: bytes) -> bytes:
    z = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_mul(ctypes.byref(z), x, y)
    return bytes(z)

def scalar_invert(s: bytes) -> bytes:
    inv = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_invert(ctypes.byref(inv), s)
    return bytes(inv)

def scalar_to_point(s: bytes) -> bytes:
    p = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255_base(ctypes.byref(p), s)
    return bytes(p)

def point_mul(scalar: bytes, point: bytes) -> bytes:
    out = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255(ctypes.byref(out), scalar, point)
    return bytes(out)

def point_add(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_add(ctypes.byref(r), p, q)
    return bytes(r)

def point_sub(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_sub(ctypes.byref(r), p, q)
    return bytes(r)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


def scalar_negate(s: bytes) -> bytes:
    """
    Computes the mathematical negation of a scalar modulo the curve order.
    Returns -s mod q.
    """
    neg = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_negate(ctypes.byref(neg), s)
    return bytes(neg)


"""
NOTE: this is super important:
    Since we are adding a paderson commitment now, we need to ensure 

"""

L = (1 << 252) + 27742317777372353535851937790883648493

DALEK_H = python_bulletproofs.get_dalek_default_h()

SCALAR_ONE = b'\x01' + b'\x00' * 31

def int_to_scalar_bytes(val: int) -> bytes:
    """Wraps integers safely around the Curve25519 order and serializes to 32 bytes."""
    L = (1 << 252) + 27742317777372353535851937790883648493
    scalar_int = val % L
    return scalar_int.to_bytes(32, byteorder='little')


# this is important for OPRF's HASH inside!!!!
def hash_point_to_scalar(point_bytes: bytes) -> int:
    h = hashlib.sha256(point_bytes).digest()
    return int.from_bytes(h, 'little') % L

## 1.1 Benchmark on Each EC-operation ##

In [4]:
def run_microbenchmarks(iterations: int = 30) -> pd.DataFrame:
    """
    Measures the average execution time of libsodium EC wrapper functions.
    """
    print(f"=== Starting Microbenchmarks ({iterations} iterations per function) ===")
    
    # 1. Pre-generate valid inputs so we don't accidentally time the setup
    # (Assuming your wrappers are already loaded and working)
    s1 = random_scalar()
    s2 = random_scalar()
    p1 = random_point()
    p2 = random_point()
    
    # Ristretto255 operates on 32-byte chunks, so we create 32 random bytes for XOR
    b1 = os.urandom(32)
    b2 = os.urandom(32)

    # 2. Define the functions to test and their required arguments
    tests = [
        ("random_scalar", random_scalar, ()),
        ("Fixed-Based Multiplication", scalar_to_point, (s1,)),
        ("Variable-Based Multiplication", point_mul, (s1, p1)),
        ("EC Addition", point_add, (p1, p2)),
        ("EC Subtraction", point_sub, (p1, p2))
    ]

    results = []

    # 3. Run the benchmarks
    for func_name, func, args in tests:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            func(*args)
            end = time.perf_counter()
            
            times.append(end - start)
            
        # Calculate the average time
        avg_time_sec = sum(times) / iterations
        
        results.append({
            "Function": func_name,
            "Avg Time (Seconds)": avg_time_sec,
            "Avg Time (Microseconds)": avg_time_sec * 1_000_000
        })

    # 4. Format into a pandas DataFrame and sort by slowest to fastest
    df = pd.DataFrame(results)
    df = df.sort_values(by="Avg Time (Microseconds)", ascending=False).reset_index(drop=True)
    
    return df


# Run the 30-iteration benchmark
benchmark_df = run_microbenchmarks(iterations=30)
# Print the beautifully formatted DataFrame
print("\nBenchmark Results (Sorted from slowest to fastest):")
print(benchmark_df.to_string(index=False))

=== Starting Microbenchmarks (30 iterations per function) ===

Benchmark Results (Sorted from slowest to fastest):
                     Function  Avg Time (Seconds)  Avg Time (Microseconds)
Variable-Based Multiplication            0.000155               154.620000
   Fixed-Based Multiplication            0.000054                53.973333
               EC Subtraction            0.000043                43.040000
                  EC Addition            0.000038                37.893333
                random_scalar            0.000010                10.093333


# 2. Workflow for Client Getting OPRF (with proof) #

## 2.1 Client Side Functionality ##

In [5]:
class FaceAuthenticationClient:
    def __init__(self, facial_vector: np.ndarray, hypervectors: np.ndarray):
        self.facial_vector = facial_vector
        self.hypervectors = hypervectors
        
        self.data_bits = 12
        self.proof_bits = 16
        self.M = (1 << (self.data_bits - 1)) - 1  # 2047 shift for 12-bit
        

#---------------------------------------Saved Computation Results -----------------------------------------------------
        self.face_commitments = []
        self.face_blinding_factors = [] # NEW: We must track the initial blinders!
        self.face_proofs = []

        self.dot_commitments = []
        self.dot_blinding_factors = [] # NEW: Tracking the homomorphic blinders

        self.bit_commitments = []
        self.bit_blinding_factors = []

        self.linkage_commitments = []
        self.linkage_proofs = []

        self.oprf_output= None # For initialization to be None 

#---------------------------------------Saved Computation Results (end) -----------------------------------------------------


        # Pre-compute plaintext LSH bits for the OPRF phase later
        self.raw_dot_products = np.dot(self.hypervectors, self.facial_vector)

        # 0 if Positive, 1 if Negative (Triggers the shift!)
        self.hashed_bits = [0 if dot >= 0 else 1 for dot in self.raw_dot_products]
    

    def CommitFace(self) -> Tuple[List[bytes], List[bytes]]:
        """
        Step 1: Locks the shifted facial vector inside standard Pedersen Commitments (Base G)
        using Python-controlled random blinders so we can track them.
        """
        
        for i, val in enumerate(self.facial_vector):
            shifted_val = int(val) + self.M
            
            if shifted_val < 0 or shifted_val >= (1 << self.data_bits):
                raise ValueError(f"Fatal: Vector value at index {i} is out of bounds.")
            
            # 1. Generate our own random blinder in Python
            r_face = random_scalar()
            self.face_blinding_factors.append(r_face)
            
            # 2. Force the Rust prover to use OUR blinder
            comm, proof = python_bulletproofs.prove_range_with_blinder(
                shifted_val, 
                self.proof_bits, 
                r_face
            )
            
            self.face_commitments.append(comm)
            self.face_proofs.append(proof)
            
        return self.face_commitments, self.face_proofs

    def CreateDotCommitment(self) -> List[bytes]:
        """
        Step 2: Homomorphically computes the dot product of the face commitments 
        and the hypervectors, AND tracks the homomorphic blinders.
        """
        if not self.face_commitments:
            raise ValueError("[Client] Error: Run CommitFace() before CreateDotCommitment()!")
        
        # Curve order L for safely wrapping the blinder math
        L = (1 << 252) + 27742317777372353535851937790883648493

        for row_idx, h_row in enumerate(self.hypervectors):
            dot_product_point = None
            r_dot_int = 0 # Running sum for the blinder
            
            # 1. Compute the homomorphic sum: sum( H_{i,j} * C_j )
            for col_idx in range(len(h_row)):
                h_val = int(h_row[col_idx])
                
                # --- Point Math ---
                C_j = self.face_commitments[col_idx]
                h_scalar = int_to_scalar_bytes(h_val) 
                scaled_C = point_mul(h_scalar, C_j)
                
                if dot_product_point is None:
                    dot_product_point = scaled_C
                else:
                    dot_product_point = point_add(dot_product_point, scaled_C)
                    
                # --- Blinder Math ---
                # We homomorphically calculate r_dot = sum(h_val * r_face) mod L
                r_face_int = int.from_bytes(self.face_blinding_factors[col_idx], 'little')
                r_dot_int = (r_dot_int + h_val * r_face_int) % L
            
            # 2. Compute the offset scalar: M * sum(H_{i,j})
            sum_h = sum(int(h) for h in h_row)
            offset_scalar_bytes = int_to_scalar_bytes(self.M * sum_h)
            
            # 3. Create the offset point using the standard Basepoint G
            offset_point = scalar_to_point(offset_scalar_bytes)
            
            # 4. Subtract the offset to get the clean dot product commitment
            final_dot_commitment = point_sub(dot_product_point, offset_point)
            
            # Save the point AND the calculated blinder!
            self.dot_commitments.append(final_dot_commitment)
            self.dot_blinding_factors.append(r_dot_int.to_bytes(32, byteorder='little'))

        return self.dot_commitments
    
    def CreateBitCommitment(self, GLOBAL_H: bytes) -> List[bytes]:
        """
        Step 3: Creates Pedersen commitments for the extracted LSH bits (0 or 1).
        Formula: C_bit = b_i * G + r_i * H
        """
        
        for b_val in self.hashed_bits:
            # 1. Generate a fresh, mathematically secure random scalar (r_i)
            r_i = random_scalar()
            self.bit_blinding_factors.append(r_i)
            
            # 2. Compute the value point: b_i * G
            # (If b_val is 0, this results in the identity point. If 1, it results in G)
            b_scalar = int_to_scalar_bytes(b_val)
            val_point = scalar_to_point(b_scalar) # Uses standard basepoint G
            
            # 3. Compute the blinding point: r_i * H
            # We use your generated GLOBAL_H to ensure it is perfectly orthogonal to G
            blind_point = point_mul(r_i, GLOBAL_H)
            
            # 4. Add them together: C_bit = (b_i * G) + (r_i * H)
            c_bit = point_add(val_point, blind_point)
            
            self.bit_commitments.append(c_bit)
            
        return self.bit_commitments

    def CreateLinkageProof(self) -> Tuple[List[bytes], List[bytes]]:
        """
        Step 4: Computes C_test = C_d + 2^32 * C_b, and generates a 32-bit 
        Bulletproof to prove the sign bit correctly matches the dot product.
        """
        if not hasattr(self, 'dot_commitments') or not hasattr(self, 'bit_commitments'):
            raise ValueError("[Client] Error: Must compute dot and bit commitments first!")
        
        # Pre-compute the 2^32 shift
        scalar_shift_int = 1 << 32
        scalar_shift_bytes = int_to_scalar_bytes(scalar_shift_int)
        
        # The Curve25519 order (L) for safely wrapping blinders
        L = (1 << 252) + 27742317777372353535851937790883648493

        for i in range(len(self.dot_commitments)):
            C_d = self.dot_commitments[i]
            C_b = self.bit_commitments[i]
            
            # --- 1. Homomorphic Commitment Math ---
            shift_point = point_mul(scalar_shift_bytes, C_b)
            C_test = point_add(C_d, shift_point)
            self.linkage_commitments.append(C_test)
            
            # --- 2. Calculate the True Plaintext Value (v_test) ---
            true_dot = int(self.raw_dot_products[i])
            bit_val = self.hashed_bits[i]
            
            # v_test = d_i + (b_i * 2^32)
            # If honesty is maintained, this stays perfectly within [0, 2^32 - 1]
            v_test = true_dot + (bit_val << 32)
            
            # --- 3. Calculate the Exact Composite Blinder (r_test) ---
            # Extract the raw integer blinders we saved earlier
            r_d_int = int.from_bytes(self.dot_blinding_factors[i], 'little')
            r_b_int = int.from_bytes(self.bit_blinding_factors[i], 'little')
            
            # r_test = (r_dot + 2^32 * r_bit) mod L
            r_test_int = (r_d_int + scalar_shift_int * r_b_int) % L
            r_test_bytes = r_test_int.to_bytes(32, byteorder='little')
            
            # --- 4. Generate the 32-bit Proof! ---
            # We pass our composite blinder into Rust so the proof mathematically binds to C_test
            _, proof = python_bulletproofs.prove_range_with_blinder(
                v_test, 
                32, 
                r_test_bytes
            )
            
            self.linkage_proofs.append(proof)

        return self.linkage_commitments, self.linkage_proofs
    

    def EvaluateOPRF(self, R_list: List[bytes], S0_list: List[int], S1_list: List[int], G_out: bytes) -> bytes:
        """
        Client Step 5: Evaluates the OT tuples to resolve the OPRF output without 
        revealing which bits were extracted.
        """
        
        M = len(self.hashed_bits)
        obtained_product = 1
        
        for i in range(M):
            b_i = self.hashed_bits[i]
            r_i_bytes = self.bit_blinding_factors[i]
            R_i = R_list[i]
            
            # 1. Compute the shared secret point: r_i * R_i
            # Because R_i = rou_i * H, this perfectly equals rou_i * r_i * H
            shared_point = point_mul(r_i_bytes, R_i)
            shared_hash = hash_point_to_scalar(shared_point)
            
            # 2. Extract the payload based on the bit
            if b_i == 0:
                # S_i0 = Hash + delta_i  =>  delta_i = S_i0 - Hash
                extracted_val = (S0_list[i] - shared_hash) % L
            else: # b_i == 1
                # S_i1 = Hash + delta_i * k_i  =>  delta_i * k_i = S_i1 - Hash
                extracted_val = (S1_list[i] - shared_hash) % L
                
            # 3. Multiply into the running product modulo L
            obtained_product = (obtained_product * extracted_val) % L
            
        # --- Final OPRF Output ---
        # Output = G_out + (obtained_product * G)
        product_scalar_bytes = int_to_scalar_bytes(obtained_product)
        product_point = scalar_to_point(product_scalar_bytes)
        
        final_oprf_point = point_add(G_out, product_point)
        
        self.oprf_output = final_oprf_point
        return final_oprf_point

## 2.3 Server Side Functionality ##

In [6]:
class FaceAuthenticationServer:
    def __init__(self, hypervectors: np.ndarray):
        """
        Initializes the server with the public LSH hypervectors.
        Sets up the exact same mathematical bounds as the client.
        """

        self.hypervectors = hypervectors
        
        # Cryptographic configuration (Must match client perfectly)
        self.data_bits = 12
        self.proof_bits = 16
        self.M = (1 << (self.data_bits - 1)) - 1  # 2047 shift for 12-bit
        
# -------------------------------------------- Server State Storage ------------------------------------------------------
        self.client_face_commitments = []
        self.dot_commitments = []
        self.bit_commitments =[]

# -------------------------------------------- Server State Storage (end) ------------------------------------------------------

        # Initialize the OPRF Server's key and shifts
        self.k_0 = int.from_bytes(random_scalar(), 'little')
        self.k_keys = [int.from_bytes(random_scalar(), 'little') for _ in range(self.M)]


    def VerifyFaceCommitments(self, commitments: List[bytes], proofs: List[bytes]) -> bool:
        """
        Server Step 1: Receives the face commitments and 16-bit range proofs from the client.
        Verifies that the hidden face vector values strictly fit within the 12-bit shifted bounds.
        """
        if len(commitments) != len(proofs):
            print("[Server] ERROR: Number of commitments does not match number of proofs.")
            return False
            
        print(f"\n[Server] Verifying {len(commitments)} Bulletproofs (16-bit capacity)...")
        
        for i in range(len(commitments)):
            # verify_range automatically uses the Dalek default basepoints (G and H)
            # This perfectly matches what the client used in CommitFace()
            is_valid = python_bulletproofs.verify_range(commitments[i], proofs[i], self.proof_bits)
            
            if not is_valid:
                print(f"[Server] 🚨 REJECTED: Zero-Knowledge Proof at index {i} failed!")
                return False
                
        print("[Server] ✅ ACCEPTED: All face vector commitments cryptographically verified!")
        
        # Save the valid commitments into the Server's state so we can use them later
        self.client_face_commitments = commitments
        
        return True
    
    def CreateDotCommitment(self) -> List[bytes]:
        """
        Server Step 2: Homomorphically computes the dot product of the verified 
        face commitments and the hypervectors, stripping the offset perfectly.
        """
        if not self.client_face_commitments:
            raise ValueError("[Server] Error: Must run VerifyFaceCommitments successfully first!")

        self.dot_commitments = []

        for row_idx, h_row in enumerate(self.hypervectors):
            dot_product_point = None
            
            # 1. Compute the homomorphic sum: sum( H_{i,j} * C_j )
            for col_idx in range(len(h_row)):
                h_val = int(h_row[col_idx])
                C_j = self.client_face_commitments[col_idx]
                
                h_scalar = int_to_scalar_bytes(h_val) 
                scaled_C = point_mul(h_scalar, C_j)
                
                if dot_product_point is None:
                    dot_product_point = scaled_C
                else:
                    dot_product_point = point_add(dot_product_point, scaled_C)
            
            # 2. Compute the offset scalar: M * sum(H_{i,j})
            sum_h = sum(int(h) for h in h_row)
            offset_scalar_bytes = int_to_scalar_bytes(self.M * sum_h)
            
            # 3. Create the offset point using the standard Basepoint G
            offset_point = scalar_to_point(offset_scalar_bytes)
            
            # 4. Subtract the offset to get the clean dot product commitment
            final_dot_commitment = point_sub(dot_product_point, offset_point)
            
            self.dot_commitments.append(final_dot_commitment)

        return self.dot_commitments
    
    def ReceiveBitCommitments(self, bit_commitments: List[bytes]):
        """
        Server Step 3: Receives the LSH bit commitments (C_b) from the client.
        The server does NOT verify these yet, it just stores them for the linkage step.
        """
        if len(bit_commitments) != len(self.hypervectors):
            raise ValueError("[Server] Error: Number of bit commitments does not match hyperplanes.")
            
        self.bit_commitments = bit_commitments
        print(f"\n[Server] ✅ Received and saved {len(self.bit_commitments)} Bit Commitments from the client.")

    def VerifyLinkageProofs(self, linkage_proofs: List[bytes]) -> bool:
        """
        Server Step 4: Homomorphically computes the linkage commitment C_test = C_d + 2^32 * C_b.
        Then, uses the client's provided 32-bit proofs to verify the math holds true.
        """
            
        if len(linkage_proofs) != len(self.dot_commitments):
            print("[Server] 🚨 ERROR: Number of proofs does not match number of commitments.")
            return False

        print(f"\n[Server] Computing Linkage Commitments and verifying {len(linkage_proofs)} 32-bit ZKPs...")
        
        # Pre-compute the 2^32 shift scalar for the curve math
        scalar_shift_bytes = int_to_scalar_bytes(1 << 32)
        
        for i in range(len(self.dot_commitments)):
            C_d = self.dot_commitments[i]
            C_b = self.bit_commitments[i]
            
            # --- 1. Server independently computes 2^32 * C_b ---
            shift_point = point_mul(scalar_shift_bytes, C_b)
            
            # --- 2. Server independently computes C_test = C_d + (2^32 * C_b) ---
            # This is the genius step: The server builds the commitment itself,
            # guaranteeing it is perfectly bound to the true face data.
            C_test = point_add(C_d, shift_point)
            
            # --- 3. Verify the client's 32-bit proof against the SERVER'S computed commitment ---
            # We can use the standard verify_range here because the verifier in Bulletproofs
            # does not need to know the blinder, it only needs the final commitment!
            is_valid = python_bulletproofs.verify_range(C_test, linkage_proofs[i], 32)
            
            if not is_valid:
                print(f"[Server] 🚨 REJECTED: Linkage Proof at index {i} failed! The client lied about their LSH bit.")
                return False
                
        print("[Server] ✅ ACCEPTED: All Linkage Proofs verified! The LSH bits mathematically match the true dot products.")
        return True
    
    def GenerateOPRFData(self, DALEK_H: bytes) -> tuple:

        # this is to fit the board's description of number of hyperplanes... 
        M = len(self.bit_commitments)
        
        R_list, S0_list, S1_list = [], [], []
        
        # We need the standard basepoint G to compute (C_bi - G)
        G_point = scalar_to_point(int_to_scalar_bytes(1))
        
        delta_product = 1

        for i in range(M):
            C_bi = self.bit_commitments[i]
            k_i = self.k_keys[i]
            
            # Sample random scalars rou_i and delta_i
            rou_i_bytes = random_scalar()
            delta_i = int.from_bytes(random_scalar(), 'little')
            
            # Keep a running product of all deltas modulo L
            delta_product = (delta_product * delta_i) % L
            
            # --- R_i = rou_i * H ---
            R_i = point_mul(rou_i_bytes, DALEK_H)
            R_list.append(R_i)
            
            # --- S_{i,0} Math ---
            # P0 = rou_i * C_bi
            P0 = point_mul(rou_i_bytes, C_bi)
            hash0 = hash_point_to_scalar(P0)
            S_i0 = (hash0 + delta_i) % L
            S0_list.append(S_i0)
            
            # --- S_{i,1} Math ---
            # P1 = rou_i * (C_bi - G)
            C_bi_minus_G = point_sub(C_bi, G_point)
            P1 = point_mul(rou_i_bytes, C_bi_minus_G)
            hash1 = hash_point_to_scalar(P1)
            S_i1 = (hash1 + (delta_i * k_i) % L) % L
            S1_list.append(S_i1)
            
        # --- Prepare G_out = (k_0 - PI(delta_i)) * G ---
        offset_scalar_int = (self.k_0 - delta_product) % L
        G_out = scalar_to_point(int_to_scalar_bytes(offset_scalar_int))
        
        print(f"[Server] ✅ OT Setup Complete. Sending {M} tuples to Client.")
        return R_list, S0_list, S1_list, G_out

# 2. Experiment and Sanity Checks #

## 2.0 Helper Functions and Facial Dataset Load ##

Load Facial Dataset

In [7]:
# Just in case we have additional dataset to test on
folder_extracted="train"

# define the official data structure in RAM 

facial_data=dict()

def restore(obj):
    if isinstance(obj, list):
        # If it's a list of numbers, convert to numpy array
        if all(isinstance(x, (int, float)) for x in obj):
            return np.array(obj)
        # Otherwise recurse element-wise
        return [restore(x) for x in obj]
    if isinstance(obj, dict):
        return {k: restore(v) for k, v in obj.items()}
    return obj

with open(f"data_{folder_extracted}.json", "r") as f:
    facial_data_loaded = json.load(f)

facial_data= restore(facial_data_loaded)

Raw Quantization to certain bits (For 2.2 section)

In [8]:
def quantize_to_12bit(float_vector: np.ndarray) -> np.ndarray:
    """
    Safely scales a float vector [-1.0, 1.0] to a 12-bit signed integer vector [-2047, 2047].
    """
    # Clip to ensure no outliers break the math
    clipped = np.clip(float_vector, -1.0, 1.0)
    # Scale by 2047 and round to nearest integer
    quantized = np.round(clipped * 2047).astype(int)
    return quantized

VECTOR_DIM = 128
NUM_HYPERPLANES = 64

## 2.1 Whole routine Testing (Without OPRF) ##

Experiment Enviornment Setup

In [9]:
# Generate dummy float vectors [-1.0, 1.0] to simulate the neural net output
raw_face = facial_data[list(facial_data.keys())[7]][3]
raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))

# Quantize them to 12-bit
quantized_face = quantize_to_12bit(raw_face)
quantized_hyper = quantize_to_12bit(raw_hyper)

print(f"Quantized Face Vector: {quantized_face[:5]}... (Length: {len(quantized_face)})")
print(f"Quantized Hyperplanes: {len(quantized_hyper)} rows of length {len(quantized_hyper[0])}")

Quantized Face Vector: [-218  143  265 -129 -225]... (Length: 128)
Quantized Hyperplanes: 64 rows of length 128


Sanity Check for One Round of Client-Server Interaction

In [ ]:
print("\n=== 2. INITIALIZATION ===")
client = FaceAuthenticationClient(quantized_face, quantized_hyper)
server = FaceAuthenticationServer(quantized_hyper)


# --- 3. Execute the Zero-Knowledge Protocol ---

print("\n=== 3. PROTOCOL EXECUTION ===")

# ---------------------------------------------------------
# STEP 1: Commit to the Face Vector
# ---------------------------------------------------------
print("\n>>> STEP 1: Client proving valid 12-bit face data...")
start_time = time.time()
face_comms, face_proofs = client.CommitFace()
print(f"Client time: {time.time() - start_time:.2f}s")

# Network Transfer: Client -> Server
is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs)
if not is_face_valid:
    raise RuntimeError("Protocol Failed: Server rejected face commitments.")

# ---------------------------------------------------------
# STEP 2: Homomorphic Dot Products
# ---------------------------------------------------------
print("\n>>> STEP 2: Both parties homomorphically computing LSH dot products...")
# Client side
start_time = time.time()
client.CreateDotCommitment()
print(f"Client time: {time.time() - start_time:.2f}s")

# Server side
start_time = time.time()
server.CreateDotCommitment()
print(f"Server time: {time.time() - start_time:.2f}s")

# ---------------------------------------------------------
# STEP 3: Bit Commitments
# ---------------------------------------------------------
print("\n>>> STEP 3: Client extracting and committing to LSH Hash Bits...")
start_time = time.time()
bit_comms = client.CreateBitCommitment(DALEK_H)
print(f"Client time: {time.time() - start_time:.2f}s")

# Network Transfer: Client -> Server
server.ReceiveBitCommitments(bit_comms)

# ---------------------------------------------------------
# STEP 4: Linkage Proofs (The 32-bit Shift)
# ---------------------------------------------------------
print("\n>>> STEP 4: Client proving LSH bits match the encrypted dot products...")
start_time = time.time()
linkage_comms, linkage_proofs = client.CreateLinkageProof()
print(f"Client time: {time.time() - start_time:.2f}s")

# Network Transfer: Client -> Server
# The server only needs the proofs, it builds the commitments itself!
is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)

if is_linkage_valid:
    print("\n🎉 SUCCESS! END-TO-END ZERO-KNOWLEDGE LSH COMPLETED! 🎉")
    print("The server holds cryptographically verified LSH Bit Commitments!")
else:
    print("\n❌ PROTOCOL FAILED AT STEP 4.")

Timing test For CLIENT-SERVER functions

In [14]:
def run_zklsh_benchmark(facial_data_dict: dict, num_iterations: int = 10, output_file: str = "BulletProof_Routine_benchmarks.csv"):
    print(f"\n🚀 Starting Zero-Knowledge LSH Benchmark ({num_iterations} iterations)...")
    
    VECTOR_DIM = 128
    NUM_HYPERPLANES = 64
    
    # We will track timings in a dictionary of lists
    timings = {
        "Client_1_CommitFace": [],
        "Server_1_VerifyFace": [],
        "Client_2_DotCommitment": [],
        "Server_2_DotCommitment": [],
        "Client_3_BitCommitment": [],
        "Server_3_ReceiveBits": [],
        "Client_4_LinkageProof": [],
        "Server_4_VerifyLinkage": [],
        "Total_Client_Time": [],
        "Total_Server_Time": [],
        "Total_Protocol_Time": []
    }

    person_keys = list(facial_data_dict.keys())

    for i in range(num_iterations):
        print(f"   -> Running iteration {i+1}/{num_iterations}...")
        
        # --- 1. Fresh Data Setup ---
        # Grab a random person and a random face from their array
        random_person = random.choice(person_keys)
        raw_face = random.choice(facial_data_dict[random_person])
        raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))
        
        quantized_face = quantize_to_12bit(raw_face)
        quantized_hyper = quantize_to_12bit(raw_hyper)
        
        client = FaceAuthenticationClient(quantized_face, quantized_hyper)
        server = FaceAuthenticationServer(quantized_hyper)
        
        client_time_total = 0.0
        server_time_total = 0.0

        # --- STEP 1 ---
        t0 = time.perf_counter()
        face_comms, face_proofs = client.CommitFace()
        t_client1 = time.perf_counter() - t0
        client_time_total += t_client1
        timings["Client_1_CommitFace"].append(t_client1)

        t0 = time.perf_counter()
        is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs)
        t_server1 = time.perf_counter() - t0
        server_time_total += t_server1
        timings["Server_1_VerifyFace"].append(t_server1)
        
        if not is_face_valid: raise RuntimeError("Protocol Failed at Step 1.")

        # --- STEP 2 ---
        t0 = time.perf_counter()
        client.CreateDotCommitment()
        t_client2 = time.perf_counter() - t0
        client_time_total += t_client2
        timings["Client_2_DotCommitment"].append(t_client2)

        t0 = time.perf_counter()
        server.CreateDotCommitment()
        t_server2 = time.perf_counter() - t0
        server_time_total += t_server2
        timings["Server_2_DotCommitment"].append(t_server2)

        # --- STEP 3 ---
        t0 = time.perf_counter()
        bit_comms = client.CreateBitCommitment(DALEK_H)
        t_client3 = time.perf_counter() - t0
        client_time_total += t_client3
        timings["Client_3_BitCommitment"].append(t_client3)

        t0 = time.perf_counter()
        server.ReceiveBitCommitments(bit_comms)
        t_server3 = time.perf_counter() - t0
        server_time_total += t_server3
        timings["Server_3_ReceiveBits"].append(t_server3)

        # --- STEP 4 ---
        t0 = time.perf_counter()
        linkage_comms, linkage_proofs = client.CreateLinkageProof()
        t_client4 = time.perf_counter() - t0
        client_time_total += t_client4
        timings["Client_4_LinkageProof"].append(t_client4)

        t0 = time.perf_counter()
        is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)
        t_server4 = time.perf_counter() - t0
        server_time_total += t_server4
        timings["Server_4_VerifyLinkage"].append(t_server4)
        
        if not is_linkage_valid: raise RuntimeError("Protocol Failed at Step 4.")

        # --- Totals ---
        timings["Total_Client_Time"].append(client_time_total)
        timings["Total_Server_Time"].append(server_time_total)
        timings["Total_Protocol_Time"].append(client_time_total + server_time_total)

    # --- 4. Process and Export to CSV ---
    print(f"\n📊 Benchmarking complete! Exporting results to {output_file}...")
    
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        # Write CSV Headers
        writer.writerow(["Operation", "Mean_Time_(s)", "Std_Dev_(s)", "Min_Time_(s)", "Max_Time_(s)"])
        
        for operation, times in timings.items():
            mean_time = statistics.mean(times)
            std_dev = statistics.stdev(times) if len(times) > 1 else 0.0
            min_time = min(times)
            max_time = max(times)
            
            writer.writerow([
                operation, 
                f"{mean_time:.6f}", 
                f"{std_dev:.6f}", 
                f"{min_time:.6f}", 
                f"{max_time:.6f}"
            ])
            
            # Print to console for immediate satisfaction
            print(f"{operation.ljust(30)} | Mean: {mean_time:.4f}s")

    print(f"\n✅ All done. Open '{output_file}' to view the detailed metrics.")

In [ ]:
run_zklsh_benchmark(facial_data, num_iterations=10)

## 2.2 Whole routine Testing (With OPRF) ##

In [ ]:
def run_full_protocol_benchmark(facial_data_dict: dict, num_iterations: int = 10, output_file: str = "full_protocol_benchmarks.csv"):

    VECTOR_DIM = 128
    NUM_HYPERPLANES = 64
    
    # Track timings for all 5 stages
    timings = {
        "Client_1_CommitFace": [],
        "Server_1_VerifyFace": [],
        "Client_2_DotCommitment": [],
        "Server_2_DotCommitment": [],
        "Client_3_BitCommitment": [],
        "Server_3_ReceiveBits": [],
        "Client_4_LinkageProof": [],
        "Server_4_VerifyLinkage": [],
        "Server_5_GenerateOPRF": [],  # NEW: OT Setup
        "Client_5_EvaluateOPRF": [],  # NEW: OT Eval
        "Total_Client_Time": [],
        "Total_Server_Time": [],
        "Total_Protocol_Time": []
    }

    person_keys = list(facial_data_dict.keys())

    for i in range(num_iterations):
        print(f"\n   ---> Running iteration {i+1}/{num_iterations}...")
        
        # --- 1. Fresh Data Setup ---
        random_person = random.choice(person_keys)
        person_faces = facial_data_dict[random_person]
        random_face_idx = random.randint(0, len(person_faces) - 1)
        
        raw_face = np.array(person_faces[random_face_idx]).flatten()

        raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))
        quantized_face = quantize_to_12bit(raw_face)
        quantized_hyper = quantize_to_12bit(raw_hyper)
        
        client = FaceAuthenticationClient(quantized_face, quantized_hyper)
        server = FaceAuthenticationServer(quantized_hyper)
        
        client_time_total = 0.0
        server_time_total = 0.0

        # --- STEP 1: Face Commitments ---
        t0 = time.perf_counter()
        face_comms, face_proofs = client.CommitFace()
        client_time_total += (time.perf_counter() - t0); timings["Client_1_CommitFace"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs)
        server_time_total += (time.perf_counter() - t0); timings["Server_1_VerifyFace"].append(time.perf_counter() - t0)
        if not is_face_valid: raise RuntimeError("Failed at Step 1.")

        # --- STEP 2: Dot Products ---
        t0 = time.perf_counter()
        client.CreateDotCommitment()
        client_time_total += (time.perf_counter() - t0); timings["Client_2_DotCommitment"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        server.CreateDotCommitment()
        server_time_total += (time.perf_counter() - t0); timings["Server_2_DotCommitment"].append(time.perf_counter() - t0)

        # --- STEP 3: Bit Commitments ---
        t0 = time.perf_counter()
        bit_comms = client.CreateBitCommitment(DALEK_H)
        client_time_total += (time.perf_counter() - t0); timings["Client_3_BitCommitment"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        server.ReceiveBitCommitments(bit_comms)
        server_time_total += (time.perf_counter() - t0); timings["Server_3_ReceiveBits"].append(time.perf_counter() - t0)

        # --- STEP 4: Linkage Proofs ---
        t0 = time.perf_counter()
        linkage_comms, linkage_proofs = client.CreateLinkageProof()
        client_time_total += (time.perf_counter() - t0); timings["Client_4_LinkageProof"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)
        server_time_total += (time.perf_counter() - t0); timings["Server_4_VerifyLinkage"].append(time.perf_counter() - t0)
        if not is_linkage_valid: raise RuntimeError("Failed at Step 4.")

        # --- STEP 5: OT & OPRF Phase (NEW) ---
        t0 = time.perf_counter()
        R_list, S0_list, S1_list, G_out = server.GenerateOPRFData(DALEK_H)
        server_time_total += (time.perf_counter() - t0); timings["Server_5_GenerateOPRF"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        final_oprf_point = client.EvaluateOPRF(R_list, S0_list, S1_list, G_out)
        client_time_total += (time.perf_counter() - t0); timings["Client_5_EvaluateOPRF"].append(time.perf_counter() - t0)
        
        # Print the final resulting point (just the first 16 hex chars to keep the console clean)
        print(f"      💎 Final OPRF Output: {final_oprf_point.hex()[:16]}...{final_oprf_point.hex()[-16:]}")

        # --- Totals ---
        timings["Total_Client_Time"].append(client_time_total)
        timings["Total_Server_Time"].append(server_time_total)
        timings["Total_Protocol_Time"].append(client_time_total + server_time_total)

    # --- Process and Export to CSV ---
    print(f"\n📊 Benchmarking complete! Exporting results to {output_file}...")
    
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Operation", "Mean_Time_(s)", "Mean_Time_(ms)", "Std_Dev_(s)", "Min_Time_(s)", "Max_Time_(s)"])
        
        for operation, times in timings.items():
            mean_time = statistics.mean(times)
            mean_ms = mean_time * 1000  # Convert to milliseconds for easier reading
            std_dev = statistics.stdev(times) if len(times) > 1 else 0.0
            min_time = min(times)
            max_time = max(times)
            
            writer.writerow([
                operation, 
                f"{mean_time:.6f}", 
                f"{mean_ms:.3f}", 
                f"{std_dev:.6f}", 
                f"{min_time:.6f}", 
                f"{max_time:.6f}"
            ])
            
            print(f"{operation.ljust(30)} | Mean: {mean_time:.4f}s ({mean_ms:.1f} ms)")

    print(f"\n✅ All done. Detailed metrics saved to '{output_file}'.")

In [ ]:
run_full_protocol_benchmark(facial_data, num_iterations=30)